# 04 · Zero-day detection benchmark
Calibrates thresholds on benign validation windows and scores the held-out test set. Reports window metrics, per test subset metrics, recall by attack intensity and episode detection. `TIMING = True` adds the per-ID rate term, fused as max(E, T): use it for injection / retiming threat models (can-train-and-test); keep it off for rate-preserving masquerades (ROAD).

In [ ]:
import os
import sys

# Pick the dataset BEFORE importing pipeline modules: "road" (default) or a converted
# can-train-and-test set such as "ctt_set_01" (see graph-transformer/convert_can_train_and_test.py).
os.environ["CAN_DATASET"] = os.environ.get("CAN_DATASET", "road")

# Repository root = first parent that holds graph-transformer/. All logic lives in the canonical
# modules; this notebook only calls them, so it cannot drift from the scripts.
REPO = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO, "graph-transformer")) and os.path.dirname(REPO) != REPO:
    REPO = os.path.dirname(REPO)
for d in ("explainability", "attack-detection", "graph-transformer"):
    p = os.path.join(REPO, d)
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| dataset:", os.environ["CAN_DATASET"])

In [ ]:
import evaluate
from scorer import AnomalyScorerConfig

TIMING = False
RUN_FOLDS = False   # ROAD only: needs the fold models from 03_train (RUN_FOLDS = True)
config = AnomalyScorerConfig(beta_temporal=1.0) if TIMING else None

In [ ]:
metrics = evaluate.run_evaluation(scorer_config=config, show_plot=False)

In [ ]:
keys = ("roc_auc", "pr_auc", "precision", "recall", "f1", "fpr")
print("window level :", {k: round(metrics[k], 4) for k in keys})
for sub, m in metrics.get("per_subset", {}).items():
    print(f"{sub}: ROC {m['roc_auc']:.3f} | P {m['precision']:.3f} | R {m['recall']:.3f} | FPR {m['fpr']:.3f}")

In [ ]:
from IPython.display import Image, display

cm = os.path.join(evaluate.OUTPUTS_DIR, "xai_visuals", "confusion_matrix.png")
if os.path.exists(cm):
    display(Image(filename=cm))

In [ ]:
if RUN_FOLDS:
    folds = evaluate.run_fold_evaluation(6, config)